In [1]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'qiskit==2.2.3', 'numpy==2.2.6', 'scipy==1.15.3'])

0

In [7]:
import numpy as np
names = ['A 한빛식품','B 새한제과','C 미래유통','D 동양농산','E 천일상사','S1 서울마트','S2 에이스마트','S3 삼광상회','S4 광명마트','S5 대한물산']
purchase = np.array([4800,5200,4700,5000,5300])
sale = np.array([8200,8400,8500,7900,8800])
lead_days = np.array([3,2,5,4,3])
monthly_frequency = np.array([12,10,8,9,11])
synergy = {(0,5):3,(1,7):4,(2,6):2,(3,8):3,(4,9):4}
linear = np.r_[purchase/1000, -sale/1000]
pairs = {key:-0.5*value for key,value in synergy.items()}
def checks(x):
    return {'매입 1개':bool(x[:5].sum()==1), '매출 1개':bool(x[5:].sum()==1),
            '매입 예산 5300원 이하':bool(purchase@x[:5]<=5300)}
def describe(x):
    cost, revenue = int(purchase@x[:5]), int(sale@x[5:])
    bonus = sum(value*x[i]*x[j] for (i,j),value in synergy.items())
    return {'매입비(원)':cost,'매출액(원)':revenue,'매출총이익(원)':revenue-cost,
            '시너지 점수':int(bonus),'선택 공급처 납기 합(일)':int(lead_days@x[:5]),
            '선택 고객 주문빈도 합(회/월)':int(monthly_frequency@x[5:])}

x = np.array([0,0,0,0,1,0,0,0,0,1])
raw = float(np.asarray(linear)@x + sum(v*x[a]*x[b] for (a,b),v in pairs.items()))
print("원래 목적함수:", round(raw,6))
print("제약조건:", checks(x))
print("업무 결과:", describe(x))

원래 목적함수: -5.5
제약조건: {'매입 1개': True, '매출 1개': True, '매입 예산 5300원 이하': True}
업무 결과: {'매입비(원)': 5300, '매출액(원)': 8800, '매출총이익(원)': 3500, '시너지 점수': 4, '선택 공급처 납기 합(일)': 3, '선택 고객 주문빈도 합(회/월)': 11}


In [8]:
import time
import importlib.metadata as metadata
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def solve(names, linear, pairs, groups, forbidden, checks, describe,
          p=2, maxiter=240, shots=4096):
    started = time.perf_counter()
    n = len(names)
    a = np.array(linear, dtype=float)
    b = dict(pairs)
    constant = 0.0
    # 원래 목적함수 계수의 절댓값 합에 1을 더한 페널티
    penalty = float(np.abs(a).sum() + sum(abs(v) for v in b.values()) + 1)
    for indices, target in groups:
        constant += penalty * target**2
        for i in indices:
            a[i] += penalty * (1 - 2 * target)
        for offset, i in enumerate(indices):
            for j in indices[offset + 1:]:
                b[i, j] = b.get((i, j), 0) + 2 * penalty
    for i, j in forbidden:
        if i == j:
            a[i] += penalty
        else:
            b[i, j] = b.get((i, j), 0) + penalty
    # 이진 변수 x_i=(1-Z_i)/2를 대입합니다.
    h = -a / 2
    jzz = {key: value / 4 for key, value in b.items()}
    offset = constant + a.sum()/2 + sum(b.values())/4
    for (i, j), value in b.items():
        h[i] -= value / 4
        h[j] -= value / 4
    # 상태벡터 인덱스에서 0번 큐비트는 가장 오른쪽 비트입니다.
    bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1)
    raw = bits @ np.array(linear, dtype=float)
    for (i, j), value in pairs.items():
        raw += value * bits[:, i] * bits[:, j]
    energy = constant + bits @ a
    for (i, j), value in b.items():
        energy += value * bits[:, i] * bits[:, j]
    z = 1 - 2 * bits
    ising = offset + z @ h
    for (i, j), value in jzz.items():
        ising += value * z[:, i] * z[:, j]
    assert np.allclose(energy, ising)
    feasible = np.array([all(checks(x).values()) for x in bits])
    violation = np.zeros(2**n)
    for indices, target in groups:
        violation += (bits[:, indices].sum(axis=1)-target)**2
    for i, j in forbidden:
        violation += bits[:, i]*bits[:, j]
    assert np.array_equal(violation == 0, feasible)
    assert np.allclose(energy, raw + penalty*violation)
    assert feasible.any()
    optimum = raw[feasible].min()
    optimal = feasible & np.isclose(raw, optimum, atol=1e-9, rtol=0)
    assert np.all(feasible[np.isclose(energy, energy.min())])
    scale = max(1.0, float(np.abs(h).max()), max((abs(v) for v in jzz.values()), default=0))

    def circuit(angles):
        qc = QuantumCircuit(n)
        qc.h(range(n))
        for layer in range(p):
            gamma = angles[layer]  # 감마: 비용 회로 위상 각도
            beta = angles[p+layer]  # 베타: 믹서 회전 각도
            for i in range(n):
                qc.rz(2*gamma*h[i]/scale, i)
            for (i, j), value in jzz.items():
                qc.rzz(2*gamma*value/scale, i, j)
            qc.rx(2*beta, range(n))
        return qc

    history = []
    def objective(angles):
        probability = Statevector.from_instruction(circuit(angles)).probabilities()
        value = float(probability @ energy)
        history.append(value)
        return value / scale

    results = []
    for start in [0.2, 0.6, 1.0]:
        initial = np.r_[np.linspace(start, start+0.3, p), np.linspace(0.2, 0.4, p)]
        results.append(minimize(objective, initial, method='COBYLA',
                       options={'maxiter': maxiter, 'rhobeg': 0.3, 'tol': 1e-5}))
    result = min(results, key=lambda r: r.fun)
    final = Statevector.from_instruction(circuit(result.x))
    probability = final.probabilities()
    final.seed(32)  # 입력 데이터는 고정하며 측정 표본의 시드를 지정합니다.
    counts = final.sample_counts(shots)
    observed = {int(key, 2): int(value) for key, value in counts.items()}
    mode = max(observed, key=lambda k: (observed[k], -k))
    # 실제 관측된 비트열 중 QUBO 에너지가 가장 낮은 항목을 반환합니다.
    chosen = min(observed, key=lambda k: (energy[k], -observed[k], k))
    x = bits[chosen]
    feasible_count = sum(v for k, v in observed.items() if feasible[k])
    optimal_count = sum(v for k, v in observed.items() if optimal[k])
    print('버전:', {k: metadata.version(k) for k in ['qiskit', 'numpy', 'scipy']})
    print('변수 순서:', names)
    print('큐비트/층수/평가횟수:', n, p, len(history))
    print('페널티/에너지 스케일:', round(penalty, 4), round(scale, 4))
    print('초기/최종 기대 에너지:', round(history[0], 6), round(float(probability@energy), 6))
    print('감마/베타:', np.round(result.x, 5).tolist())
    print('최빈 측정 비트/제약 충족:', format(mode, f'0{n}b'), bool(feasible[mode]))
    print('반환 비트(q 높은 번호부터):', format(chosen, f'0{n}b'))
    print('선택 비트(변수 순서):', x.tolist())
    print('선택 항목:', [names[i] for i in range(n) if x[i]])
    print('원래 목적함수:', round(float(raw[chosen]), 6))
    print('선택 QUBO 에너지:', round(float(energy[chosen]), 6))
    print('제약조건:', checks(x))
    print('정확 최적값:', round(float(optimum), 6))
    print('최적값 차이:', round(float(raw[chosen]-optimum), 6) if feasible[chosen] else '실행 불가 해: 차이 평가 제외')
    print('실행 가능/최적해 관측 비율(%):', round(100*feasible_count/shots, 3), round(100*optimal_count/shots, 3))
    print('실행 가능/최적해 이론 확률(%):', round(100*probability[feasible].sum(), 3), round(100*probability[optimal].sum(), 3))
    print('선택 비트 관측횟수/이론 확률:', observed[chosen], round(float(probability[chosen]), 6))
    print('업무 결과:', describe(x))
    print('최적화 종료:', result.message)
    print('경과시간(초):', round(time.perf_counter()-started, 3))


names = ['A 한빛식품','B 새한제과','C 미래유통','D 동양농산','E 천일상사','S1 서울마트','S2 에이스마트','S3 삼광상회','S4 광명마트','S5 대한물산']
purchase = np.array([4800,5200,4700,5000,5300])
sale = np.array([8200,8400,8500,7900,8800])
lead_days = np.array([3,2,5,4,3])
monthly_frequency = np.array([12,10,8,9,11])
synergy = {(0,5):3,(1,7):4,(2,6):2,(3,8):3,(4,9):4}
linear = np.r_[purchase/1000, -sale/1000]
pairs = {key:-0.5*value for key,value in synergy.items()}
def checks(x):
    return {'매입 1개':bool(x[:5].sum()==1), '매출 1개':bool(x[5:].sum()==1),
            '매입 예산 5300원 이하':bool(purchase@x[:5]<=5300)}
def describe(x):
    cost, revenue = int(purchase@x[:5]), int(sale@x[5:])
    bonus = sum(value*x[i]*x[j] for (i,j),value in synergy.items())
    return {'매입비(원)':cost,'매출액(원)':revenue,'매출총이익(원)':revenue-cost,
            '시너지 점수':int(bonus),'선택 공급처 납기 합(일)':int(lead_days@x[:5]),
            '선택 고객 주문빈도 합(회/월)':int(monthly_frequency@x[5:])}
solve(names, linear, pairs, [(list(range(5)),1),(list(range(5,10)),1)], [], checks, describe)

버전: {'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3'}
변수 순서: ['A 한빛식품', 'B 새한제과', 'C 미래유통', 'D 동양농산', 'E 천일상사', 'S1 서울마트', 'S2 에이스마트', 'S3 삼광상회', 'S4 광명마트', 'S5 대한물산']
큐비트/층수/평가횟수: 10 2 715
페널티/에너지 스케일: 75.8 115.85
초기/최종 기대 에너지: 1481.37357 46.817249
감마/베타: [0.27406, 0.78877, -0.48392, -0.20824]
최빈 측정 비트/제약 충족: 0000000000 False
반환 비트(q 높은 번호부터): 1000010000
선택 비트(변수 순서): [0, 0, 0, 0, 1, 0, 0, 0, 0, 1]
선택 항목: ['E 천일상사', 'S5 대한물산']
원래 목적함수: -5.5
선택 QUBO 에너지: -5.5
제약조건: {'매입 1개': True, '매출 1개': True, '매입 예산 5300원 이하': True}
정확 최적값: -5.5
최적값 차이: 0.0
실행 가능/최적해 관측 비율(%): 48.193 1.904
실행 가능/최적해 이론 확률(%): 48.548 1.979
선택 비트 관측횟수/이론 확률: 78 0.019789
업무 결과: {'매입비(원)': 5300, '매출액(원)': 8800, '매출총이익(원)': 3500, '시너지 점수': 4, '선택 공급처 납기 합(일)': 3, '선택 고객 주문빈도 합(회/월)': 11}
최적화 종료: Maximum number of function evaluations has been exceeded.
경과시간(초): 5.106


정확 최적값은 −5.5이며 E와 S5의 거래입니다. 매입비는 5,300원, 매출액은 8,800원, 매출총이익은 3,500원, 시너지는 4점입니다. 시너지의 목적함수 보상은 회계상 현금 수익에 더하지 않습니다. 공급처 또는 고객사가 0개인 비트열은 해당 선택 제약을 위반합니다.

반환 해의 최적값 차이가 0이면 관측 후보에서 정확 최적해를 얻은 것입니다. 최적해 관측 비율은 전체 측정 중 그 조건을 만족한 비트열의 비중입니다. 기대 에너지에는 실행 불가 상태의 페널티도 확률에 따라 반영됩니다. 최빈 비트열이 제약을 위반해도 관측 분포 안에 실행 가능한 해가 있을 수 있습니다. 실행시간에는 계수 구성, 완전탐색 검증, 시뮬레이션, 최적화와 최종 표본 추출이 포함됩니다.